# 🤖 OpenAI API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **OpenAI API 키**를 입력하고,
**프롬프트를 작성**해서 LLM의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, GPT!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 메시지로 역할 부여 (role prompting) |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** https://platform.openai.com/api-keys 에서 발급받으세요.
> 키는 `sk-...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 OpenAI 공식 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [ ]:
# OpenAI 라이브러리 설치
%pip install openai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import openai
    print("✅ 설치 완료 · openai", openai.__version__)
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

✅ 설치 완료 · openai 2.43.0


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [1]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요 (sk-...): ")

print("✅ API 키가 등록되었습니다.")

OpenAI API Key를 입력하세요 (sk-...): ··········
✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`client`는 우리가 OpenAI에게 요청을 보내는 **통로**입니다.
위에서 등록한 환경변수(`OPENAI_API_KEY`)를 자동으로 읽어옵니다.

> ⚠️ **모델 선택 주의**
> - `gpt-4o-mini` · `gpt-4.1-mini` : 일반(비추론형) 모델 → 이 실습 코드가 **그대로 동작**합니다. (추천)
> - `gpt-5.4-mini` · `gpt-5.4-nano` 등 **GPT-5 계열은 추론형**이라 `temperature`를 바꿀 수 없고(1 고정),
>   `max_tokens` 대신 `max_completion_tokens`를 써야 합니다. (아래 `ask()` 함수가 알아서 처리하지만,
>   6단계의 temperature 비교 실습 결과는 달라지지 않습니다.)

In [2]:
from openai import OpenAI

client = OpenAI()   # 환경변수에서 API 키를 자동으로 읽어옵니다

# 사용할 모델 이름 (원하면 아래 중 하나로 바꿔보세요)
#   "gpt-4o-mini"    → 가장 저렴한 일반 모델 (실습에 추천 · 기본값)
#   "gpt-4.1-mini"   → 조금 더 성능이 좋은 일반 모델
#   "gpt-5.4-mini"   → 최신 추론형 모델 (temperature 고정 주의)
#   "gpt-5.4"        → 상위 추론형 모델 (비용 높음)
MODEL = "gpt-4o-mini"

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gpt-4o-mini


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
client.chat.completions.create(
    model=MODEL,          # 어떤 모델을 쓸지
    messages=[            # 대화 내용 (list)
        {"role": "user", "content": "질문 내용"}
    ]
)
```

- `messages`는 **대화 목록**입니다. `role`이 `"user"`면 사람, `"assistant"`면 GPT, `"system"`이면 역할 지시입니다.
- 답변 텍스트는 `response.choices[0].message.content` 안에 들어 있습니다.
- 답변 길이(`max_tokens`)·창의성(`temperature`) 같은 옵션은 6단계에서 다룹니다.

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요! 당신을 한 문장으로 소개해 주세요."}
    ]
)

# 답변 텍스트 꺼내기
print(response.choices[0].message.content)

안녕하세요! 저는 다양한 주제에 대한 질문에 답하고 정보를 제공하는 인공지능 챗봇입니다.


### 응답 안에는 뭐가 들어있을까?

답변 말고도 **몇 개의 토큰을 썼는지** 같은 정보가 함께 옵니다.
토큰은 대략적인 '단어 조각' 단위이고, 요금이 토큰 수에 따라 매겨지므로 알아두면 좋습니다.

In [4]:
print("입력 토큰 수 :", response.usage.prompt_tokens)
print("출력 토큰 수 :", response.usage.completion_tokens)
print("멈춘 이유    :", response.choices[0].finish_reason)

입력 토큰 수 : 20
출력 토큰 수 : 25
멈춘 이유    : stop


## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 대화 맨 앞에 역할 지시 메시지가 추가됩니다.
> GPT-5 계열 모델도 에러 없이 돌아가도록 파라미터 이름을 자동으로 맞춰 줍니다.

In [5]:
# 메세지 형식
'''
client.chat.completions.create(
    model="gpt-4o-mini",          # ← 최상위 인자
    temperature=1.0,              # ← 최상위 인자
    max_tokens=1024,              # ← 최상위 인자
    messages=[                    # ← 최상위 인자 (리스트)
        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)
        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)
    ]
)
'''

'\nclient.chat.completions.create(\n    model="gpt-4o-mini",          # ← 최상위 인자\n    temperature=1.0,              # ← 최상위 인자\n    max_tokens=1024,              # ← 최상위 인자\n    messages=[                    # ← 최상위 인자 (리스트)\n        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)\n        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)\n    ]\n)\n'

In [6]:
def ask(prompt, system=None, max_tokens=1024, temperature=1.0):
    """프롬프트를 보내고 LLM의 답변 텍스트를 돌려줍니다."""
    messages = []
    if system:                                       # 역할(system) 지시가 있으면 맨 앞에 추가
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": MODEL, "messages": messages}
    if MODEL.startswith("gpt-5"):                     # GPT-5 계열(추론형)은 파라미터가 다름
        kwargs["max_completion_tokens"] = max_tokens  #  · max_tokens 대신 이것을 사용
        #  · temperature는 기본값(1)만 지원하므로 넣지 않음
    else:                                            # gpt-4o / gpt-4.1 계열(일반)
        kwargs["max_tokens"] = max_tokens
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ ask() 함수 준비 완료")

✅ ask() 함수 준비 완료


👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [7]:
my_prompt = "파이썬 리스트와 튜플의 차이를 초보자에게 설명해줘. 예시 코드도 하나 보여줘."

print(ask(my_prompt))

리스트와 튜플은 파이썬에서 데이터를 저장하는 두 가지 컬렉션 타입입니다. 이 두 가지의 주요 차이점은 다음과 같습니다:

1. **변경 가능성 (Mutability)**:
   - **리스트(list)**: 변경 가능(mutable)합니다. 즉, 리스트에 저장된 값을 변경, 추가, 삭제할 수 있습니다.
   - **튜플(tuple)**: 변경 불가능(immutable)합니다. 즉, 튜플에 저장된 값을 변경하거나 추가, 삭제할 수 없습니다.

2. **표기법**:
   - 리스트는 대괄호 `[]`로 감싸서 표현합니다.
   - 튜플은 괄호 `()`로 감싸서 표현합니다.

3. **성능**:
   - 튜플은 리스트보다 메모리 사용이 더 효율적이며, 성능이 약간 더 빠릅니다. 그래서 변경되지 않는 데이터 그룹을 저장할 때 튜플을 사용하면 좋습니다.

4. **용도**:
   - 리스트는 데이터가 변경될 수 있는 경우 사용하고, 튜플은 변경되지 않는 데이터를 그룹화할 때 사용합니다.

이제 예시 코드를 보여드리겠습니다.

```python
# 리스트 예시
my_list = [1, 2, 3, 4]
print("리스트:", my_list)

# 리스트에 요소 추가
my_list.append(5)
print("리스트 수정 후:", my_list)

# 튜플 예시
my_tuple = (1, 2, 3, 4)
print("튜플:", my_tuple)

# 튜플은 수정할 수 없으므로 다음 코드는 오류가 발생합니다.
# my_tuple.append(5)  # 이 줄을 활성화하면 오류가 발생합니다.
```

위 코드에서 `my_list`는 리스트로, 요소를 추가할 수 있고, `my_tuple`은 튜플로, 요소를 추가할 수 없음을 보여줍니다.


> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [8]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

=== 역할: 엄격한 교수님 ===
재귀(recursion)는 프로그래밍 및 수학에서 함수가 자기 자신을 호출하는 프로세스를 의미합니다. 이는 특정 문제를 더 작은 부분 문제로 나누어 해결하는 기법으로, 각 호출이 문제를 점진적으로 단순화하는 역할을 합니다. 

재귀 함수는 일반적으로 두 가지 주요 요소로 구성됩니다. 첫 번째는 **기저 조건(base case)**으로, 이는 더 이상 재귀 호출이 이루어지지 않고 함수를 종료해야 하는 조건을 정의합니다. 두 번째는 **재귀 조건(recursive case)**으로, 여기서는 함수가 자신을 재호출하면서 문제의 크기를 줄이는 방식으로 정의됩니다.

예를 들어, 팩토리얼(factorial) 함수는 재귀를 이용하여 정의할 수 있습니다:

- 기저 조건: 0! = 1
- 재귀 조건: n! = n × (n - 1)!

이와 같은 방식으로 재귀는 알고리즘을 간결하고 직관적으로 표현할 수 있는 강력한 도구입니다. 그러나 재귀 사용 시, 특히 깊은 재귀 호출이 발생할 경우 스택 오버플로우(stack overflow)와 같은 문제가 발생할 수 있으므로, 적절한 기저 조건을 설정하고 재귀 호출의 깊이를 관리하는 것이 중요합니다.

=== 역할: 친근한 선배 ===
재귀는 말 그대로 자기 자신을 불러오는 걸 말해. 쉽게 비유를 들어볼게. 

생각해봐, 양파를 깎는다고 할 때. 양파를 깎다보면, 껍질을 한 겹 벗기고 나면 또 그 아래에 새로운 양파가 나오는 거야. 그러니까, "양파를 깎는 과정"이 다시 한번 반복되는 느낌이야. 이게 바로 재귀야! 

프로그래밍에서 재귀는 어떤 함수가 자기 자신을 다시 호출하는 방식으로, 문제를 작은 문제로 나눠서 해결하는데 유용해. 예를 들어, 팩토리얼(정수 n의 곱)을 구하는 함수를 생각해보면, 
n! = n × (n-1)! 이라고 정의할 수 있어. 여기서 (n-1)!이 다시 자기 자신을 호출하는 거니까 재귀인 거지. 

이해되었지? 필요하면 더 이야기해줄게!


> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [35]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

[Zero-shot]
부정


In [36]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정: 긍정
"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
문장: 내 친구가 나를 배신했어. 너무 슬퍼.
감정: 부정

문장: 새로운 프로젝트가 너무 기대돼!
감정: 긍정

문장: 오늘은 아무것도 하고 싶지 않아. 지루해.
감정: 부정

문장: 이 책은 정말 재미있어! 계속 읽고 싶어.
감정: 긍정


> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [11]:
prompt = "AI 관해서 주제로 짧은 삼행시를 지어줘."

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

=== temperature = 0.0 (거의 항상 비슷) ===
인공지능, 미래를 여는 열쇠  
지혜의 바다에서 함께 나아가  
능력의 한계를 넘어 꿈을 꿉니다.

=== temperature = 1.0 (다양하게) ===
알고리즘의 힘으로  
인공지능이 펼치는  
세상을 함께 꿈꿔요.


In [12]:
# max_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_tokens=30))

=== max_tokens = 30 (짧게 잘림) ===
인공지능(AI)의 역사는 복잡하고 다양합니다. 다음은 주요 사건과 발전 과정을 정리한 개요입니다.

###


## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [21]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [14]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라! 당신의 이름을 기억할게요. 어떻게 도와드릴까요?


In [17]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

죄송하지만, 당신의 이름을 알 수 있는 정보는 없습니다. 이름을 알려주시면 제가 기억할 수 있습니다!


> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [37]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.


## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [59]:
# 과제 1 예시 (여기서부터 직접 수정해 보세요)
guide_system = "너는 밝고 친근한 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다. 부산 사투리도 쓴다."

print(ask("부산 여행 코스를 추천해줘.", system=guide_system))

부산 여행이라면 넘 재밌겠당! 🌊✨ 그럼 몇 가지 코스를 추천해줄게!

1. **해운대 해수욕장 🏖️**  
   아침에 해운대 해수욕장에서 일출 바라보면 기분 짱 좋아! 해변가에서 산책하고, 맛있는 조개구이 먹어보꼬~ 🍽️

2. **동백섬 🌼**  
   해운대에서 가까운 동백섬은 경치가 진짜 예쁘당! 고운 꽃들도 많고, 산책하기 딱 좋지. 오르막 올라가면 멋진 풍경이 기다리고 있어~ 📸

3. **부산 아쿠아리움 🐠**  
   해운대 근처에 있는 아쿠아리움은 다양한 해양 생물들을 볼 수 있어! 어린이들도 좋아할걸? 🐬

4. **태종대 📍**  
   오후엔 태종대로 가봐야 해! 바다 절경이 끝내주고, 등대도 구경할 수 있어. 가는 길에 깔딱고개 걷는 것도 재밌지~ 🚶‍♀️

5. **자갈치 시장 🐟**  
   저녁엔 자갈치 시장에 가서 신선한 해산물 요리 먹어봐~ 싱싱한 생선에 맥주 한 잔하면 여행 완전 갑이지!🍻

이렇게 하루 코스를 해보면 부산의 매력을 쏙 느낄 수 있을 거야! ✈️ 부산에 온 김에 즐거운 시간 보내고 가라잉! 🥳


In [20]:
# 과제 2 · 과제 3 은 아래 빈 셀에서 자유롭게 작성해 보세요.

# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 한글 단어를 영어 단어로 번역해줘.

한글 단어: 행복
영어 단어: Happy

한글 단어: 사과
영어 단어: Apple

한글 단어: 컴퓨터
영어 단어: Computer
"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
한글 단어: 사랑  
영어 단어: Love  

한글 단어: 학교  
영어 단어: School  

한글 단어: 친구  
영어 단어: Friend  


In [50]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라! 반가워요. 어떻게 도와드릴까요?


In [51]:
print(chat("난 지금 학교에 있어. 이것도 기억해줘."))

알겠어요, 코알라. 지금 학교에 있군요! 학교 생활은 어떤지 궁금해요. 도움이 필요하면 언제든지 말해 주세요!


In [52]:
print(chat("곧 있으면 점심 시간이야 학교 급식 먹으러 갈거야."))

점심 시간이 곧 오네요! 학교 급식은 어떤 메뉴인지 궁금해요. 맛있는 점심 드시고 좋은 시간 보내세요!


In [53]:
print(chat("이제 내이름이랑 내가 어디있는지 곧 뭐먹을건지 맞춰봐."))

좋아요! 너의 이름은 코알라이고, 지금 학교에 있으며 곧 학교 급식으로 점심을 먹으러 간다고 했지. 맞나? 맞다면 잘 기억하고 있는 거네요!


In [58]:
print(chat("굳 잘 맞추네."))

고마워요, 코알라! 여러분의 취향에 맞게 잘 맞출 수 있도록 노력하고 있어요. 또 다른 질문이나 이야기하고 싶은 것이 있다면 언제든지 말해 주세요!


In [49]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.


---

### 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `client.chat.completions.create(...)` 로 LLM에 요청을 보낸다
2. `messages` 는 `system` / `user` / `assistant` 로 이루어진 **대화 목록**이다
3. `system` 으로 **역할**을, `temperature`·`max_tokens` 로 **답변 스타일과 길이**를 조절한다
4. **few-shot 예시**로 원하는 출력 형식을 유도할 수 있다
5. 대화를 기억시키려면 **이전 기록을 매번 함께 보낸다**

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 https://platform.openai.com/api-keys 에서 삭제·재발급하세요.
